# Notebook 05 — Archetype Keys Validation & Cleanup

**Goal:** Load `archetype_mapping.csv` and `archetype_keys.csv`, validate key format,
check for duplicates/missing segments, produce clean verified outputs.


In [1]:
import pandas as pd
import numpy as np
import sys, os
import warnings
warnings.filterwarnings('ignore')

# Robust src path — works whether run manually or via papermill
_src_path = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if _src_path not in sys.path:
    sys.path.insert(0, _src_path)
from config import *

# ── Paths (notebooks run from notebooks/, outputs are in notebooks/data/outputs/)
OUT = OUT_PATH

mapping = pd.read_csv(f'{OUT}archetype_mapping.csv')
keys    = pd.read_csv(f'{OUT}archetype_keys.csv')

print(f'archetype_mapping : {mapping.shape}')
print(f'archetype_keys    : {keys.shape}')
print('\nMapping columns  :', mapping.columns.tolist())
print('Keys columns     :', keys.columns.tolist())

archetype_mapping : (2142, 9)
archetype_keys    : (96, 9)

Mapping columns  : ['Division', 'Portal', 'Size', 'bucket_min', 'bucket_max', 'ASP_bucket', 'New_Bucket', 'archetype_key', 'total_qty']
Keys columns     : ['archetype_key', 'Division', 'Portal', 'Size', 'New_Bucket', 'price_range_min', 'price_range_max', 'total_qty', 'fine_bucket_count']


## 1. Preview Data

In [2]:
print('=== archetype_mapping (first 5) ===')
print(mapping.head().to_string(index=False))
print('\n=== archetype_keys (first 10) ===')
print(keys.head(10).to_string(index=False))

=== archetype_mapping (first 5) ===
Division  Portal   Size  bucket_min  bucket_max ASP_bucket  New_Bucket archetype_key  total_qty
      BP       1 Single           0          99       0-99           1  BPTT11Single      12366
      BP       1 Single         100         199    100-199           1  BPTT11Single         94
      BP       1 Single         200         299    200-299           1  BPTT11Single       8166
      BP       1 Single         300         399    300-399           2  BPTT12Single      72159
      BP       1 Single         400         499    400-499           3  BPTT13Single      41191

=== archetype_keys (first 10) ===
archetype_key Division  Portal   Size  New_Bucket  price_range_min  price_range_max  total_qty  fine_bucket_count
 BPTT11Single       BP       1 Single           1                0              299      20626                  3
 BPTT12Single       BP       1 Single           2              300              399      72159                  1
 BPTT13Sing

## 2. Validate Key Format

Expected format: `Division + 'TT' + Portal + NewBucket + Size`  
Example: `HLTT16CABIN`


In [3]:
import re

# PORTAL_ABBREV and PORTAL_IS_INT come from config via `from config import *`

def build_expected_key(row):
    if PORTAL_IS_INT:
        abbrev = PORTAL_ABBREV.get(int(row['Portal']), str(row['Portal']))
        return f"{row['Division']}{abbrev}{row['Portal']}{row['New_Bucket']}{row['Size']}"
    else:
        return f"{row['Portal']}{row['Division']}{row['Size']}{row['New_Bucket']}"
keys['expected_key'] = keys.apply(build_expected_key, axis=1)
key_col = 'archetype_key'

mismatches = keys[keys[key_col] != keys['expected_key']]
print(f'Key format mismatches: {len(mismatches)}')
if len(mismatches):
    print(mismatches[[key_col, 'expected_key']].to_string())
else:
    print('✓ All archetype keys match expected format')

Key format mismatches: 0
✓ All archetype keys match expected format


## 3. Check No Duplicate Keys

In [4]:
dup_keys = keys[keys.duplicated(subset=[key_col], keep=False)]
print(f'Duplicate archetype_keys: {len(dup_keys)}')
if len(dup_keys):
    print(dup_keys.to_string())
else:
    print('✓ No duplicate keys')

dup_map = mapping[mapping.duplicated(subset=[key_col, 'bucket_min'], keep=False)]
print(f'\nDuplicate (key, bucket_min) rows in mapping: {len(dup_map)}')
if not len(dup_map):
    print('✓ No duplicate mapping rows')

Duplicate archetype_keys: 0
✓ No duplicate keys

Duplicate (key, bucket_min) rows in mapping: 0
✓ No duplicate mapping rows


## 4. Check All Segments Present


In [5]:
# ── Section 4: Check all segments present ─────────────────────────────────
# Dynamically reads actual segments from archetype_keys.csv.
# Works for both TT (portal=int) and EC (portal=string).
# No hardcoded SEGMENT_K — counts come from the data itself.

actual_segs = (keys.groupby(['Division', 'Portal', 'Size'])['New_Bucket']
               .nunique()
               .reset_index()
               .rename(columns={'New_Bucket': 'got_k'}))

print('=== Segment → Archetype-Bucket Count ===')
print(f'{"Segment":<30} {"Buckets":>7}')
print('-' * 40)

for _, row in actual_segs.sort_values(['Division', 'Portal', 'Size']).iterrows():
    seg_label = f"{row['Division']}_{row['Portal']}_{row['Size']}"
    print(f'{seg_label:<30} {int(row["got_k"]):>7}')

print()
print(f'Total segments found : {len(actual_segs)}')
print(f'Total archetype keys : {len(keys)}')
print()

# ── Warn if any segment has only 1 bucket (likely under-clustered) ─────────
single_bucket_segs = actual_segs[actual_segs['got_k'] == 1]
if len(single_bucket_segs):
    print(f'⚠  {len(single_bucket_segs)} segment(s) have only 1 bucket — check clustering:')
    for _, r in single_bucket_segs.iterrows():
        print(f'   {r["Division"]}_{r["Portal"]}_{r["Size"]}')
else:
    print('✓ All segments have ≥ 2 buckets')


=== Segment → Archetype-Bucket Count ===
Segment                        Buckets
----------------------------------------
BP_1_Single                          7
BS_1_Single                          4
DF_1_DF                              3
DF_1_DFT                             5
HL_1_CABIN                           8
HL_1_LARGE                           8
HL_1_MEDIUM                          8
HL_1_SO2                             6
HL_1_SO3                             8
SL_1_CABIN                           8
SL_1_LARGE                           8
SL_1_MEDIUM                          8
SL_1_SO2                             7
SL_1_SO3                             8

Total segments found : 14
Total archetype keys : 96

✓ All segments have ≥ 2 buckets


## 5. Null / Missing Value Check

In [6]:
print('=== Nulls in archetype_mapping ===')
null_map = mapping.isnull().sum()
print(null_map[null_map > 0] if null_map.any() else 'None ✓')

print('\n=== Nulls in archetype_keys ===')
null_keys = keys.isnull().sum()
print(null_keys[null_keys > 0] if null_keys.any() else 'None ✓')

=== Nulls in archetype_mapping ===
None ✓

=== Nulls in archetype_keys ===
None ✓


## 6. Summary Statistics per Archetype Key

In [7]:
print('=== Archetype Keys Summary ===')
print(f'Total keys       : {len(keys)}')
print(f'Total segments   : {keys.groupby(["Division","Portal","Size"]).ngroups}')
print(f'Mapping rows     : {len(mapping)}')
print(f'Mapping buckets  : {mapping["bucket_min"].nunique()} unique bucket_min values')
print()

print('=== Keys per Segment ===')
seg_summary = (keys.groupby(['Division','Portal','Size'])
               .agg(n_buckets=('New_Bucket','count'),
                    total_qty=('total_qty','sum'),
                    price_min=('price_range_min','min'),
                    price_max=('price_range_max','max'))
               .reset_index())
print(seg_summary.to_string(index=False))

=== Archetype Keys Summary ===
Total keys       : 96
Total segments   : 14
Mapping rows     : 2142
Mapping buckets  : 348 unique bucket_min values

=== Keys per Segment ===
Division  Portal   Size  n_buckets  total_qty  price_min  price_max
      BP       1 Single          7    1099961          0      29499
      BS       1 Single          4      10977          0       6399
      DF       1     DF          3     204270          0       8199
      DF       1    DFT          5     644317          0       6599
      HL       1  CABIN          8     655610          0       8399
      HL       1  LARGE          8      77507          0      12399
      HL       1 MEDIUM          8     397976          0      10299
      HL       1    SO2          6     293823          0      34799
      HL       1    SO3          8     594484          0      27899
      SL       1  CABIN          8     184846          0       8699
      SL       1  LARGE          8     152410          0      10899
      SL   

## 7. Drop helper column and Save Verified Output

In [8]:
keys = keys.drop(columns=['expected_key'], errors='ignore')

keys.to_csv(f'{OUT}archetype_keys.csv', index=False)
print(f'✓ Saved verified archetype_keys.csv  ({len(keys)} rows → {OUT}archetype_keys.csv)')

✓ Saved verified archetype_keys.csv  (96 rows → data/outputs/TT/archetype_keys.csv)
